## memo

- URL: https://www.kaggle.com/code/koyamaryuji/qwen-array-task-inference/output?scriptVersionId=342705803
- 正解と不正解の両方のルールのトレースのsft

In [5]:
import polars as pl
from pathlib import Path
import sys
sys.path.append(str('d:/qwen_reasoning_test/ArrayTask'))
from src.gen_task import RULES

df = pl.read_csv(Path("debug_predictions.csv"))
df = df.sort("id")
stop = 0
for pred in df.iter_rows(named=True):
    print("###" * 50)
    print(f'Task ID: \n{pred["id"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Prompt: \n{pred["raw_prompt"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Raw_output: \n{pred["raw_output"]}')
    print("===" * 50)
    print(f'Target: \n{pred["target"]}')
    print("===" * 50)
    print(f'Finish: \n{pred["finish_reason"]}')
    print("===" * 50)
    print(f'tokens: \n{pred["num_tokens"]}')
    if pred["finish_reason"] == "stop":
        stop += 1

print(stop)


######################################################################################################################################################
Task ID: 
000_0
Prompt: 

Infer the transformation rule from examples.
Output the final array.
        
Example 1

Input:
[7, 7, 2, 6, 5, 6]

Output:
[6, 5, 6, 2, 7, 7]

Example 2

Input:
[1, 8, 1, 1, 7, 4, 0, 3, 5]

Output:
[5, 3, 0, 4, 7, 1, 1, 8, 1]

Example 3

Input:
[4, 3, 2, 2, 3, 5]

Output:
[5, 3, 2, 2, 3, 4]

Example 4

Input:
[6, 3, 6, 7, 8, 4, 6]

Output:
[6, 4, 8, 7, 6, 3, 6]

Example 5

Input:
[1, 2, 0, 9, 6, 2]

Output:
[2, 6, 9, 0, 2, 1]

Query

Input:
[2, 3, 2, 0, 3, 7, 6, 2, 9]

Output:
Raw_output: 
<think>
Candidate rule: ['shift_right']

Example 0:
Input: [7, 7, 2, 6, 5, 6]
Output: [6, 7, 7, 2, 6, 5]
Expected output: [6, 5, 6, 2, 7, 7]
Match: False
    
The candidate rule is rejected because the output does not match the expected output.
Candidate rule: ['reverse']

Example 0:
Input: [7, 7, 2, 6, 5, 6]
Output: [6, 5, 6

In [6]:
import re
import ast
import polars as pl
from pathlib import Path


def extract_answer(text):
    if text is None:
        return 'NOT_FOUND'

    matches = re.findall(r'\[[^\[\]]*\]', text)
    arrays = []
    for match in matches:
        try:
            value = ast.literal_eval(match)

            if isinstance(value, list):
                arrays.append(value)

        except (ValueError, SyntaxError):
            pass

    if arrays == []:
        return 'NOT_FOUND'

    return str(arrays[-1])

df = pl.read_csv(Path("debug_predictions.csv"))

match_count = 0
match = []
miss = []
for pred in df.iter_rows(named=True):
    # print("###" * 50)
    answer = extract_answer(pred["raw_output"])
    if answer == pred["target"]:
        # print(pred["id"])
        match_count += 1
        match.append(pred["id"].split("_")[0])
    else:
        miss.append(pred["id"].split("_")[0])
        # print(pred["id"])
        # print(pred["raw_prompt"].encode().decode("unicode-escape"))
        # print(pred["target"])
print(f"len(df): {len(df)}")
print(f"match_count: {match_count}")
# print(miss)
from collections import Counter


counts = Counter(miss)
match_counts = Counter(match)

len(df): 520
match_count: 101


# 正解

In [7]:
match_results = []
for i, j in match_counts.items():
    for rule in RULES:
        if rule["id"] == i:
            # print(i, j, rule)
            match_results.append({"task_id": i, "count": j, "rule": rule["primitives"]})

match_results = sorted(match_results, key=lambda x: x["count"], reverse=True)
match_results

[{'task_id': '007', 'count': 10, 'rule': ['append_zeros']},
 {'task_id': '010', 'count': 9, 'rule': ['multiply_constant']},
 {'task_id': '006', 'count': 9, 'rule': ['prepend_zeros']},
 {'task_id': '002', 'count': 7, 'rule': ['shift_left']},
 {'task_id': '043', 'count': 7, 'rule': ['shift_left', 'append_zeros']},
 {'task_id': '004', 'count': 6, 'rule': ['rotate_left']},
 {'task_id': '001', 'count': 5, 'rule': ['shift_right']},
 {'task_id': '009', 'count': 5, 'rule': ['pop_left']},
 {'task_id': '044', 'count': 5, 'rule': ['shift_left', 'pop_right']},
 {'task_id': '008', 'count': 4, 'rule': ['pop_right']},
 {'task_id': '032', 'count': 4, 'rule': ['shift_right', 'prepend_zeros']},
 {'task_id': '013', 'count': 3, 'rule': ['sort_ascending']},
 {'task_id': '017', 'count': 3, 'rule': ['mirror']},
 {'task_id': '035', 'count': 3, 'rule': ['shift_right', 'pop_left']},
 {'task_id': '042', 'count': 3, 'rule': ['shift_left', 'prepend_zeros']},
 {'task_id': '029', 'count': 2, 'rule': ['reverse', 'mir

# 不正解

In [8]:
results = []
for i, j in counts.items():
    for rule in RULES:
        if rule["id"] == i:
            # print(i, j, rule)
            results.append({"task_id": i, "count": j, "rule": rule["primitives"]})

results = sorted(results, key=lambda x: x["count"], reverse=True)
results

[{'task_id': '022', 'count': 10, 'rule': ['reverse', 'pop_right']},
 {'task_id': '050', 'count': 10, 'rule': ['shift_left', 'pairwise_sum']},
 {'task_id': '024', 'count': 10, 'rule': ['reverse', 'multiply_constant']},
 {'task_id': '048', 'count': 10, 'rule': ['shift_left', 'modulo']},
 {'task_id': '040', 'count': 10, 'rule': ['shift_right', 'pairwise_sum']},
 {'task_id': '036', 'count': 10, 'rule': ['shift_right', 'multiply_constant']},
 {'task_id': '012', 'count': 10, 'rule': ['modulo']},
 {'task_id': '028', 'count': 10, 'rule': ['reverse', 'pairwise_sum']},
 {'task_id': '037', 'count': 10, 'rule': ['shift_right', 'add_constant']},
 {'task_id': '015', 'count': 10, 'rule': ['differences']},
 {'task_id': '026', 'count': 10, 'rule': ['reverse', 'modulo']},
 {'task_id': '031', 'count': 10, 'rule': ['reverse', 'take_odd_positions']},
 {'task_id': '030', 'count': 10, 'rule': ['reverse', 'take_even_positions']},
 {'task_id': '051', 'count': 10, 'rule': ['shift_left', 'mirror']},
 {'task_id':